# Step 6 — First model (fold 0)

**Settings: `GPU T4 x2`, `Internet On`.**

**Inputs to attach:**
1. Competition data
2. `rsna-knee-04-build-cache` (notebook output — the 8 GB pixel cache)
3. `rsna-knee-03-extract-all` (notebook output — pseudo-labels)

## Architecture

```
slice     ->  2.5D input: 3 adjacent slices as RGB channels
                  |
              EfficientNet-B0, shared across all 72 slices
                  |  (B, 3, 24, D)
series    ->  bi-GRU over the 24 slices, attention-pooled -> one vector per series
                  |  (B, 3, H)
study     ->  attention pool over sagittal/coronal/axial, masked for missing series
                  |  (B, H)
              Linear -> 12 logits
```

**Why 2.5D rather than 3D.** A meniscal tear spans several slices, so through-plane context matters — but full 3D convolutions cost roughly 4x more for a marginal gain on only 4,407 studies. Feeding three adjacent slices as colour channels buys most of the context at 2D price, and lets us use ImageNet-pretrained weights, which matter enormously at this dataset size.

**Why attention pooling over series.** Different findings live in different sequences: effusion and Baker's cysts are clearest on fluid-sensitive images, meniscal tears on sagittal, MCL on coronal. Attention lets the model weight planes per finding rather than averaging them into mush. The mask handles studies missing a plane.

## Two deliberate choices

**Soft targets.** We train on the continuous 0–100 pseudo-labels divided by 100, never thresholded. A report the extractor scored 55 is genuinely ambiguous and should contribute a weaker gradient than one scored 95. Thresholding throws that information away, and since the metric is AUC — pure ranking — nothing downstream needs hard labels either.

**The 58 gold studies are excluded from training.** They're the only unbiased measurement available. Used as a clean holdout, they tell you whether the model learned anatomy or just learned to imitate the extractor's mistakes.

## No horizontal flips

Standard augmentation for natural images, wrong here. The cache canonicalised every knee to a right-knee orientation; flipping would swap medial and lateral and silently corrupt four of the twelve labels. Affine jitter and intensity augmentation instead.

### 1 · Setup

In [ ]:
import os, gc, glob, time, math, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

ROOT  = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CACHE = glob.glob("/kaggle/input/**/cache.npy", recursive=True)[0]
PSEUDO= glob.glob("/kaggle/input/**/pseudo_labels.csv", recursive=True)[0]
GOLD  = glob.glob("/kaggle/input/**/gold58.csv", recursive=True)[0]
print("cache ", CACHE)
print("pseudo", PSEUDO)

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]

arr    = np.load(CACHE, mmap_mode="r")
pseudo = pd.read_csv(PSEUDO)
gold   = pd.read_csv(GOLD)
train  = pd.read_csv(f"{ROOT}/train.csv")

N, NSER, NSL, PX, _ = arr.shape
print(f"cache {arr.shape}  |  pseudo {pseudo.shape}  |  gold {len(gold)}")

# row order in cache follows train.csv order
assert (pseudo["StudyInstanceUID"].values == train["StudyInstanceUID"].values).all()
Y = pseudo[KEYS].values.astype(np.float32) / 100.0
print("soft target range", Y.min(), Y.max())
print("DEV:", torch.cuda.get_device_name(0))

### 2 · Splits

The 58 gold studies come out of training entirely — that's the clean holdout.

The rest split 5 ways, stratified on the binarised pseudo-labels so each fold sees a similar mix of findings. `PatientID` was verified 1:1 with `StudyInstanceUID` during EDA, so there's no patient-level leakage to guard against; plain stratification is enough.

Which series slots are actually populated gets computed once here and reused as an attention mask.

In [ ]:
gold_ids = set(gold["StudyInstanceUID"])
is_gold  = train["StudyInstanceUID"].isin(gold_ids).values
print(f"gold holdout: {is_gold.sum()}  |  trainable: {(~is_gold).sum()}")

t0 = time.time()
present = np.zeros((N, NSER), dtype=np.float32)
for i in range(N):
    for k in range(NSER):
        present[i, k] = float(arr[i, k, NSL // 2].max() > 0)
print(f"series presence computed in {time.time()-t0:.0f}s")
print("mean series per study:", present.sum(1).mean().round(2))
print("studies with zero series:", int((present.sum(1) == 0).sum()))

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    os.system("pip install -q iterative-stratification")
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

usable = np.where((~is_gold) & (present.sum(1) > 0))[0]
Ybin   = (Y[usable] > 0.5).astype(int)
mskf   = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds  = np.zeros(len(usable), dtype=int)
for f, (_, v) in enumerate(mskf.split(usable, Ybin)):
    folds[v] = f

FOLD = 0
tr_idx = usable[folds != FOLD]
va_idx = usable[folds == FOLD]
gd_idx = np.where(is_gold & (present.sum(1) > 0))[0]
print(f"\ntrain {len(tr_idx):,} | val {len(va_idx):,} | gold {len(gd_idx)}")

### 3 · Dataset

**2.5D channels.** For slice `j` we stack `[j-1, j, j+1]` *from the same series*, clamped at the boundaries. Never across series — those are different acquisitions and blending them would be meaningless.

**Augmentation** is affine jitter (small rotation, scale, translation) plus intensity gain/bias, applied consistently across every slice of a series so the volume stays coherent. No flips, for the laterality reason above.

In [ ]:
import cv2

class KneeDS(Dataset):
    def __init__(self, idx, train_mode):
        self.idx, self.train_mode = idx, train_mode

    def __len__(self): return len(self.idx)

    def __getitem__(self, n):
        i = self.idx[n]
        v = np.asarray(arr[i]).astype(np.float32) / 255.0        # (3, 24, 160, 160)

        if self.train_mode:
            ang   = np.random.uniform(-12, 12)
            scale = np.random.uniform(0.9, 1.1)
            tx    = np.random.uniform(-0.06, 0.06) * PX
            ty    = np.random.uniform(-0.06, 0.06) * PX
            M = cv2.getRotationMatrix2D((PX/2, PX/2), ang, scale)
            M[0, 2] += tx; M[1, 2] += ty
            gain = np.random.uniform(0.85, 1.15)
            bias = np.random.uniform(-0.08, 0.08)
            for s in range(NSER):
                if v[s].max() == 0: continue
                for j in range(NSL):
                    v[s, j] = cv2.warpAffine(v[s, j], M, (PX, PX), borderValue=0)
                v[s] = np.clip(v[s] * gain + bias, 0, 1)

        # 2.5D: (3 series, 24 slices, 3 adjacent, H, W)
        out = np.empty((NSER, NSL, 3, PX, PX), np.float32)
        for s in range(NSER):
            for j in range(NSL):
                lo, hi = max(j-1, 0), min(j+1, NSL-1)
                out[s, j] = np.stack([v[s, lo], v[s, j], v[s, hi]])

        return (torch.from_numpy(out),
                torch.from_numpy(present[i]),
                torch.from_numpy(Y[i]))

def loader(idx, train_mode, bs):
    return DataLoader(KneeDS(idx, train_mode), batch_size=bs, shuffle=train_mode,
                      num_workers=2, pin_memory=True, drop_last=train_mode,
                      persistent_workers=True)

d = KneeDS(tr_idx[:2], True)[0]
print("sample:", d[0].shape, d[1].shape, d[2].shape)

### 4 · Model

In [ ]:
import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):                 # x (B, T, D)
        a = self.w(x).squeeze(-1)                    # (B, T)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        a = a.softmax(-1).unsqueeze(-1)
        return (x * a).sum(1)

class Net(nn.Module):
    def __init__(self, backbone="efficientnet_b0", nlab=12, hid=256, pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))

    def forward(self, x, mask):                      # x (B, S, T, 3, H, W)
        B, S, T = x.shape[:3]
        f = self.enc(x.reshape(B * S * T, *x.shape[3:]))       # (B*S*T, D)
        f = f.reshape(B * S, T, -1)
        f, _ = self.gru(f)                                     # (B*S, T, 2H)
        f = self.slice_pool(f).reshape(B, S, -1)               # (B, S, 2H)
        f = self.ser_pool(f, mask)                             # (B, 2H)
        return self.head(f)

model = Net().cuda()
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

### 5 · Train

`BCEWithLogitsLoss` against soft targets. Mixed precision, cosine schedule with warmup, gradient checkpointing off (batch is small enough).

Every epoch reports **three** numbers:

- **val AUC (pseudo)** — against held-out pseudo-labels. Optimises well but shares the extractor's biases.
- **gold AUC (58)** — against expert labels. Noisy at n=58, but it's the only unbiased signal.
- **per-label** on the gold set, so you can see which findings are actually being learned.

If val-pseudo climbs while gold stagnates, the model is learning the extractor rather than the anatomy. That's the failure mode worth watching for.

In [ ]:
EPOCHS, BS, LR = 8, 6, 3e-4

tl = loader(tr_idx, True,  BS)
vl = loader(va_idx, False, BS)
gl = loader(gd_idx, False, BS)

opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps  = EPOCHS * len(tl)
sched  = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")
lossf  = nn.BCEWithLogitsLoss()

@torch.no_grad()
def evaluate(dl):
    model.eval(); P, T = [], []
    for x, m, y in dl:
        with torch.amp.autocast("cuda"):
            p = model(x.cuda(non_blocking=True), m.cuda())
        P.append(p.float().sigmoid().cpu().numpy()); T.append(y.numpy())
    return np.concatenate(P), np.concatenate(T)

def macro_auc(P, T, thresh=0.5):
    out = []
    for c in range(T.shape[1]):
        y = (T[:, c] > thresh).astype(int)
        if y.min() == y.max(): out.append(np.nan); continue
        out.append(roc_auc_score(y, P[:, c]))
    return np.nanmean(out), out

best = 0
for ep in range(EPOCHS):
    model.train(); tot = 0; t0 = time.time()
    for bi, (x, m, y) in enumerate(tl):
        x, m, y = x.cuda(non_blocking=True), m.cuda(), y.cuda()
        with torch.amp.autocast("cuda"):
            loss = lossf(model(x, m), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
        if bi % 100 == 0:
            el = time.time() - t0
            print(f"  ep{ep} {bi}/{len(tl)} loss {tot/(bi+1):.4f} "
                  f"eta {(len(tl)-bi)*el/(bi+1)/60:.0f}m", flush=True)

    Pv, Tv = evaluate(vl); av, _   = macro_auc(Pv, Tv)
    Pg, Tg = evaluate(gl); ag, per = macro_auc(Pg, Tg, thresh=0.5)
    print(f"EPOCH {ep}  loss {tot/len(tl):.4f}  val(pseudo) {av:.4f}  gold(58) {ag:.4f}"
          f"  [{(time.time()-t0)/60:.0f} min]", flush=True)

    if av > best:
        best = av
        torch.save({"model": model.state_dict(), "fold": FOLD,
                    "val_auc": av, "gold_auc": ag, "epoch": ep},
                   "/kaggle/working/model_fold0.pt")
        print("   saved")

print(f"\nbest val(pseudo) AUC: {best:.4f}")
print("\nper-label on gold holdout (final epoch):")
for l, a in zip(LABELS, per):
    print(f"  {l:<18} {a:.3f}")

### 6 · Save out-of-fold predictions

Needed later for ensembling — blending models on raw leaderboard feedback is how people overfit the public split. Out-of-fold predictions let you tune the blend offline against your own validation.

In [ ]:
oof = pd.DataFrame(Pv, columns=LABELS)
oof.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values[va_idx])
oof.to_csv("/kaggle/working/oof_fold0.csv", index=False)

gp = pd.DataFrame(Pg, columns=LABELS)
gp.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values[gd_idx])
gp.to_csv("/kaggle/working/gold_preds_fold0.csv", index=False)

print("wrote oof_fold0.csv, gold_preds_fold0.csv, model_fold0.pt")
print(f"checkpoint size: {os.path.getsize('/kaggle/working/model_fold0.pt')/1e6:.0f} MB")